# Method C: every criterion from measurements, no expert ratings

**Idea (option 4):** compute C1–C5 and C10 from the lab data, fill the river (C6/C7) and groundwater (C8/C9) criteria from measured data and ML, and combine with **fixed weights chosen once** (equal or data-driven). No expert rates an industry; the paper's expert scores are used only to *compare*.

| Criterion | Source |
|---|---|
| C1–C5, C10 | lab data, as in method B (`wqml.summarise`) |
| C6 / C7 river impact, pre / post | limit-weighted pollutant load (concentration / limit × toxicity × flow). A drain-ratio *scenario* is shown separately; it is an assumption, not a measured delivered fraction |
| C8 / C9 groundwater impact, pre / post | exploratory ML model of groundwater contamination per locality and season, trained on well samples (NIH 2012, CGWB March 2012) with one common marker set |

**Limits, stated up front:** wells have village names, not coordinates, so groundwater impact is per *locality*: industries in the same locality get the same C8/C9. Arsenic is left out: the report itself attributes it to natural floodplain conditions.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import wqml
from wqml import CRITERIA, WORSE_IF_HIGH
warnings.filterwarnings("ignore")

raw, SOURCE = wqml.load()
assert SOURCE != "dummy", "needs the paper's source data in data/uppcb/"
df = wqml.prepare(raw)
summary = wqml.summarise(df)
labels = wqml.paper_labels()
reg = pd.read_csv(wqml.UPPCB / "stations.csv")
place = reg[reg.role == "outlet"].drop_duplicates("source_id").set_index("source_id")[["locality", "drain"]]
summary = summary.join(place)
summary[["industrial_area", "industry_type", "locality", "drain"]]

## 1. Limit-weighted load per industry and season
For each outlet sample: Σ over pollutants of (concentration / CPCB limit) × toxicity factor × flow (MLD). This is the pollution an industry sends out, in "limit-equivalent" units, so BOD, chromium and fluoride can be added fairly.

In [ ]:
o = df.dropna(subset=["outlet", "limit"]).copy()
o["leq"] = o["outlet"] / o["limit"] * o["pollutant"].map(wqml.TOXICITY).fillna(1) * o["discharge_flow_mld"]
load = o.pivot_table(index="source_id", columns="season", values="leq", aggfunc=lambda v: v.sum(min_count=1))  # no flow -> unknown, not 0
load = load.reindex(summary.index)[["pre_monsoon", "post_monsoon"]]
load.round(2).join(summary[["industrial_area", "locality"]]).sort_values(["industrial_area", "post_monsoon"])

## 2. C6 / C7 river impact
**Main value:** the limit-weighted load itself (no factor).

**Scenario only (assumption, not a measurement):** BOD at the drain's mouth (CPCB 2018–2023, median) ÷ BOD in the drain near the industries (2012 samples: Loni SW-7, City Jail SW-13), capped at 1. This is a ratio of concentrations from different years and places, with no flow data, so it is *not* the fraction of load that reaches the Ganga. It is shown only to test how much such an assumption would change the ranking.

In [ ]:
drains = pd.read_csv("data/cpcb/drains_unnao.csv")
mouth = drains.dropna(subset=["bod_mgL"]).query("year >= 2018").groupby("drain")["bod_mgL"].median()
near = df.query("pollutant == 'BOD'").set_index("source_id")["outlet"]
near = {"Loni": near.loc["site2-S6"].mean(), "City Jail": near.loc["magarwara1-S1"].mean()}  # SW-7, SW-13
DRAIN_RATIO = (mouth / pd.Series(near)).clip(upper=1)
print(pd.DataFrame({"BOD at Ganga mouth (median 2018-23)": mouth, "BOD in drain 2012": pd.Series(near),
                    "scenario ratio (assumed)": DRAIN_RATIO}).round(2))
summary["C6"], summary["C7"] = load["pre_monsoon"], load["post_monsoon"]  # main: load, no factor
ratio = summary["drain"].map(DRAIN_RATIO).fillna(DRAIN_RATIO.mean())     # unknown drain -> average (scenario only)
scenario = summary.assign(C6=summary["C6"] * ratio, C7=summary["C7"] * ratio)
summary[["industrial_area", "drain", "C6", "C7"]].round(3)

## 3. Groundwater samples
Two surveys from the source report with the **same marker set**: NIH 2012 (Tables 7.28a/b) and CGWB March 2012 (Annex V). The CGWB April 2010 survey (Annex III) has metals only, so it is left out. Each well gets a locality: from the well register (NIH) or by village name (CGWB; unmatched villages are dropped). Villages far from industry (Makanpur, Antwa, Bhudda…) form a **background** group with zero industrial load.

Contamination index per sample = mean of (value / drinking-water limit, IS 10500) for **fluoride (1.5 mg/L) and TDS (2000 mg/L; from EC × 0.64 when TDS is missing)** — the two markers measured in both surveys. Chromium is not used: the surveys measured different chromium forms (Cr(VI) vs total).

In [ ]:
U = wqml.UPPCB
num = lambda s: pd.to_numeric(s.replace({"ND": 0, "BDL": 0}), errors="coerce")
VILLAGE = {  # CGWB village names -> locality of the NIH well register (Table 5.2)
    "laukhera": "Unnao", "lauikhera": "Unnao", "jalimkhera": "Unnao", "jalim": "Unnao", "dakari": "Unnao",
    "rishinagar": "Unnao", "risinagar": "Unnao", "subasnagar": "Unnao", "ambikapuram": "Unnao", "awas": "Unnao",
    "avas": "Unnao", "krishna sweet": "Unnao", "maduwasi": "Maswasi", "maswari": "Maswasi", "maswasi": "Maswasi",
    "benthar": "Banthar", "bauther": "Banthar", "cept": "Banthar", "darmkanta": "Ind. Area Site-1",
    "daramkanta": "Ind. Area Site-1", "shivnagar": "Ind. Area Site-1", "dahi": "Ind. Area Site-2",
    "jhanjhari": "Ind. Area Site-2", "jhayhari": "Ind. Area Site-2", "jhauyhari": "Ind. Area Site-2",
    "adargh": "Magarwara", "adarsh": "Magarwara", "sekhpur": "Magarwara", "shekhpur": "Magarwara",
    "makanpur": "background", "katari": "background", "ganghut": "background", "melalam": "background",
    "munnipurwa": "background", "bhudda": "background", "antwa": "background", "naubatganj": "background"}
WELL_TYPES = r"IM|HP|DW|TW|SP|DTW"
def village(loc):
    return next((v for k, v in VILLAGE.items() if k in loc.lower()), None)

# NIH 2012 (Tables 7.28a/b) + well register
w = pd.read_csv(U / "wells.csv").set_index("well_id")
w.loc[w["location"].str.contains("Banthar|Leather Technology", case=False), "locality"] = "Banthar"
g = pd.read_csv(U / "groundwater.csv", dtype=str)
nih = pd.DataFrame({"locality": g["station_id"].map(w["locality"]), "season": g["season"],
                    "depth_ft": g["station_id"].map(w["depth_ft"]), "survey": "NIH 2012",
                    "Cr": num(g["Cr6"]), "F": num(g["F"]), "TDS": num(g["TDS"]), "Cl": num(g["Cl"]), "SO4": num(g["SO4"])})
# CGWB March 2012 (Annex V): Cr in ug/L
a5 = pd.read_csv(U / "annex5_cgwb_2012.csv", dtype=str)
a5 = a5[a5["type"].str.contains(WELL_TYPES)]
cg12 = pd.DataFrame({"locality": a5["location"].map(village), "season": "pre_monsoon", "depth_ft": np.nan,
                     "survey": "CGWB Mar 2012", "Cr": num(a5["Cr_ugL"]).fillna(num(a5["Cr6_ugL"])) / 1000,
                     "F": num(a5["F"]), "TDS": num(a5["EC"]) * 0.64})
gw = pd.concat([nih, cg12], ignore_index=True).dropna(subset=["locality"])
DRINK = {"F": 1.5, "TDS": 2000}
gw["index"] = pd.concat([gw[k] / v for k, v in DRINK.items()], axis=1).mean(axis=1)
gw = gw.dropna(subset=["index"])
print(len(gw), "well samples with a locality")
gw.groupby(["locality", "season"])["index"].agg(["size", "median"]).round(2)

## 4. Exploratory ML: groundwater contamination from industrial load
Target: log of the contamination index. Features: total limit-weighted load of the **plants and CETPs** in the well's locality (drains are left out because they carry those same plants' effluent), well depth, season, survey. The model must predict localities it has not seen (e.g. Jamuka), so it is tested **leave-one-locality-out**, and the model choice itself is nested inside that test. Exploratory: few localities, one year.

In [ ]:
plants = summary.index[~summary["industry_type"].isin(["drain", "sewage_drain"])]
locload = load.loc[plants].join(summary["locality"]).groupby("locality")[["pre_monsoon", "post_monsoon"]].sum(min_count=1)
def loc_load(loc, season):
    return 0.0 if loc == "background" else locload[season].get(loc, np.nan)
gw["loc_load"] = [loc_load(l, s) for l, s in zip(gw["locality"], gw["season"])]
gw = gw.dropna(subset=["loc_load"])
def features(t):
    return pd.DataFrame({"log_load": np.log1p(t["loc_load"]), "depth_ft": t["depth_ft"].astype(float),
                         "pre": (t["season"] == "pre_monsoon").astype(float),
                         "cgwb_2012": (t["survey"] == "CGWB Mar 2012").astype(float)}, index=t.index)
Xg, yg, groups = features(gw), np.log(gw["index"].clip(lower=1e-3)), gw["locality"]
GW_MODELS = {"always average": DummyRegressor(), "ridge": RidgeCV(alphas=np.logspace(-2, 3, 30)),
             "k-nearest neighbours": KNeighborsRegressor(7),
             "random forest": RandomForestRegressor(300, min_samples_leaf=5, random_state=0, n_jobs=-1),
             "gradient boosting": GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=0)}
gpipe = lambda e: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), e)
logo = LeaveOneGroupOut()
# each model, leave-one-locality-out
cv = {n: cross_val_predict(gpipe(e), Xg, yg, groups=groups, cv=logo) for n, e in GW_MODELS.items()}
gres = pd.DataFrame({n: {"MAE (log index)": abs(p - yg).mean(),
                         "spearman": pd.Series(p, index=yg.index).corr(yg, method="spearman")} for n, p in cv.items()}).T
# nested: choose the model on the other localities, then predict the held-out one
nested = pd.Series(np.nan, index=yg.index)
for tr, te in logo.split(Xg, yg, groups):
    inner = {n: abs(cross_val_predict(gpipe(e), Xg.iloc[tr], yg.iloc[tr], groups=groups.iloc[tr], cv=LeaveOneGroupOut()) - yg.iloc[tr]).mean()
             for n, e in GW_MODELS.items()}
    best = min(inner, key=inner.get)
    nested.iloc[te] = gpipe(GW_MODELS[best]).fit(Xg.iloc[tr], yg.iloc[tr]).predict(Xg.iloc[te])
gres.loc["nested choice"] = [abs(nested - yg).mean(), nested.corr(yg, method="spearman")]
gres.sort_values("MAE (log index)").round(3)

In [ ]:
GW_CHOSEN = gres.drop("nested choice")["MAE (log index)"].idxmin()
signal = gres.loc["nested choice", "MAE (log index)"] < gres.loc["always average", "MAE (log index)"]
print("Chosen:", GW_CHOSEN, "| nested test beats always-average:", signal)
gmodel = gpipe(GW_MODELS[GW_CHOSEN]).fit(Xg, yg)
# predicted contamination index per industry's locality and season (typical well: median depth, NIH survey)
for crit, season in [("C8", "pre_monsoon"), ("C9", "post_monsoon")]:
    tt = pd.DataFrame({"loc_load": [loc_load(l, season) for l in summary["locality"]], "season": season,
                       "depth_ft": gw["depth_ft"].astype(float).median(), "survey": "NIH 2012"}, index=summary.index)
    summary[crit] = np.exp(gmodel.predict(features(tt))) if signal else np.nan  # no signal -> leave neutral
summary[["industrial_area", "locality", "C8", "C9"]].round(3)

## 5. Combine with weights chosen once
Same within-area ranking as method B (`wqml.local_weights`: measured ranks stretched so their mean is neutral, missing = neutral). Weights: **equal** and **CRITIC** (from the data's spread and correlation). The paper's expert weights only as a reference.

In [ ]:
def rank_score(s, w):
    score = (wqml.local_weights(s) * pd.Series(w)[CRITERIA]).sum(axis=1)
    return score / score.groupby(s["industrial_area"]).transform("mean")

used = [c for c in CRITERIA if summary[c].notna().sum() >= 10]
G = pd.DataFrame({c: (-summary[c] if WORSE_IF_HIGH[c] else summary[c]) for c in used})
G = G.fillna(G.median())
G = (G - G.min()) / (G.max() - G.min())
critic = G.std() * (1 - G.corr()).sum()
WEIGHTS = {
    "equal": {c: (1 / len(used) if c in used else 0) for c in CRITERIA},
    "CRITIC (data)": {c: (critic[c] / critic.sum() if c in used else 0) for c in CRITERIA},
    "paper expert (reference)": pd.read_csv(wqml.PAPER / "criteria.csv").set_index("criterion")["weight"].to_dict(),
}
print("criteria with data:", used)
pd.DataFrame(WEIGHTS).round(3)

## 6. Compare with the experts' ranking
Spearman within each area (1 = same order as the paper's Table 10). *B* = method B as it is (C6/C7 only where upstream/downstream were sampled, C8/C9 empty). *drain-ratio scenario* multiplies C6/C7 by the assumed drain ratio of section 2.

In [ ]:
t = summary[["industrial_area"]].join(labels)
b_summary = wqml.summarise(df)  # method B's own criteria
def agree(s):
    return wqml.agreement(t.assign(S=s), "S")
rows = {}
for name, w in WEIGHTS.items():
    rows[f"C, {name}"] = agree(rank_score(summary, w))
    rows[f"B, {name}"] = agree(rank_score(b_summary, w))
rows["C, equal, drain-ratio scenario"] = agree(rank_score(scenario.assign(C8=summary["C8"], C9=summary["C9"]), WEIGHTS["equal"]))
pd.DataFrame(rows).T.round(3)

In [ ]:
final = summary[["industrial_area", "locality"]].copy()
final["C_score"] = rank_score(summary, WEIGHTS["equal"])
final["C_rank"] = final.groupby("industrial_area")["C_score"].rank(method="min").astype(int)  # 1 = most polluting
final = final.join(labels[["fuzzy_weight"]])
final["fuzzy_rank"] = final.groupby("industrial_area")["fuzzy_weight"].rank(method="min").astype(int)
final.sort_values(["industrial_area", "C_rank"]).round(3)

## 7. Mass balance: do the plant loads add up to the drain loads?
Load (kg/day) = concentration (mg/L) × flow (MLD). Sum over the plants and CETPs that discharge to each drain (2012, mean of both rounds, plants with a known flow), against the drain's own load: the 2012 drain samples (Loni SW-7 downstream, City Jail SW-13) × the 2012 drain flows (report Table 3.3), and the 2016 joint survey. Chromium marks tannery effluent; BOD and COD also carry town sewage (about 5 MLD into Loni, 15 MLD into City Jail).

In [ ]:
o = df.dropna(subset=["outlet"]).join(place[["drain"]].rename(columns={"drain": "to_drain"}), on="source_id")
pl = o[~o["industry_type"].isin(["drain", "sewage_drain"])].assign(load=lambda d: d["outlet"] * d["discharge_flow_mld"])
plant = pl.dropna(subset=["load"]).groupby(["to_drain", "pollutant", "source_id"])["load"].mean().groupby(["to_drain", "pollutant"]).sum()
d16 = pd.read_csv("data/geo/drains_2016_cpcb.csv").set_index("drain")
rows = []
for drain, sid, q12, k16 in [("Loni", "site2-S6", 10.12, "Loni"), ("City Jail", "magarwara1-S1", 22.3, "City Jail (Dakary/Dakari)")]:
    c12 = df[df["source_id"] == sid].groupby("pollutant")["outlet"].mean()
    r16 = d16.loc[k16]
    for p, col in [("Cr", "Cr_total_mg_L"), ("BOD", "BOD_mg_L"), ("COD", "COD_mg_L")]:
        rows.append({"drain": drain, "pollutant": p, "plants 2012 (kg/d)": plant.get((drain, p), np.nan),
                     "drain 2012 (kg/d)": c12.get(p, np.nan) * q12, "drain 2016 (kg/d)": float(r16[col]) * float(r16["flow_2016_MLD"])})
mb = pd.DataFrame(rows)
mb["share explained 2012"] = mb["plants 2012 (kg/d)"] / mb["drain 2012 (kg/d)"]
print("plants with a known flow per drain:", pl.dropna(subset=["load"]).groupby("to_drain")["source_id"].nunique().to_dict())
mb.round(2)

## Notes
- Placeholders that need expert sign-off: `wqml.TOXICITY`, `wqml.C5_CLASS`, the drinking-water marker set (F, TDS), the village → locality map.
- The source report gives village coordinates (§3.6); wells and outfalls still need their own. Better groundwater impact needs well and outfall **coordinates**: then the model can use distance to each outfall and give each industry its own C8/C9.
- River context (not used for ranking, it cannot separate industries): `data/uppcb/river_ganga.csv`, CWC monthly Kanpur / Bhitaura 2001–2011.